[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C57_Small_Object_Detection_Course/02_architecture/02_architecture_small.ipynb)

# 02 · 架构层面的解法：分辨率、多尺度与上下文（FPN 层级分配 / P2 成本 / 加权融合 / 可变形采样）

目标：把「小目标要不要加 P2」这类问题从<strong>争论</strong>变成<strong>计算</strong>。
本 notebook 全程纯 numpy，所有结论都以 `assert` 固定下来。

**你会亲手实现：**
1. 针孔模型下的 TSR 尺寸分布，并**解析地证明**尺寸密度 ∝ 1/s²（不是数据集偏差，是几何必然）
2. FPN 的层级分配公式 `k = k0 + floor(log2(sqrt(wh)/224))`，并统计 TSR 分布下各层的负载
3. 「一个目标在某 stride 的网格上有几个中心点」的解析式与蒙特卡洛验证
4. 加 P2 的 FLOPs / 显存增量计算器（精确到 85/21 这个分数），以及四种降本手段
5. FPN top-down / PAN bottom-up / BiFPN fast normalized fusion，用**脉冲实验**证明 FPN 单向性
6. 可变形采样点在小目标上的覆盖率分析

> 心智模型：**小目标在架构层面上，八成是分辨率问题。
> 「目标的像素数」决定信息量上界，「目标占的格子数」决定网络能否表示它——
> 提分辨率两项都动，加 P2 只动第二项，换主干两项都不动。**

## 1 · TSR 尺寸分布：从针孔模型推出来，而不是从数据集统计出来

In [ ]:
import numpy as np, math, json
rng = np.random.default_rng(57)

def focal_px(width_px, hfov_deg):
    '''针孔模型：f = (W/2) / tan(HFOV/2)，单位是像素。'''
    return (width_px / 2.0) / math.tan(math.radians(hfov_deg) / 2.0)

W_IMG, H_IMG, HFOV = 1920, 1080, 60.0
F_PX  = focal_px(W_IMG, HFOV)
SIGN_M = 0.60                      # 中国圆形限速牌直径 60 cm

def sign_px(dist_m, sign_m=SIGN_M, f=F_PX):
    '''标志在图像上的像素边长： s = f * S / Z'''
    return f * sign_m / np.asarray(dist_m, dtype=float)

print(f'相机: {W_IMG}x{H_IMG}, HFOV={HFOV}deg  ->  f = {F_PX:.2f} px')
print(f'60 cm 限速牌的像素尺寸:')
for z in [10, 20, 30, 50, 60, 80, 100, 130, 150]:
    s = float(sign_px(z))
    tag = '  <- COCO small (<32px)' if s < 32 else ''
    print(f'   {z:4d} m  ->  {s:6.2f} px{tag}')

assert abs(F_PX - 1662.77) < 0.05,  F_PX
assert abs(float(sign_px(50)) - 19.95) < 0.05
assert float(sign_px(100)) < 12.0
print('\n✅ 50 m 外 ~20 px, 100 m 外 ~10 px —— 这就是 TSR 的物理起点。')

In [ ]:
# 距离均匀采样 -> 尺寸密度 ∝ 1/s^2 （s = c/Z, Z~U(a,b) => p(s) = c / ((b-a) s^2)）
Z_MIN, Z_MAX, N_SAMP = 8.0, 150.0, 400_000
C_GEO = F_PX * SIGN_M                       # s = C_GEO / Z

Z = rng.uniform(Z_MIN, Z_MAX, N_SAMP)
S = C_GEO / Z                               # 每个实例的像素边长

def frac_below_analytic(thr):
    '''P(s < thr) = P(Z > C/thr)，解析式。'''
    z_star = np.clip(C_GEO / thr, Z_MIN, Z_MAX)
    return (Z_MAX - z_star) / (Z_MAX - Z_MIN)

print(f'尺寸范围: {S.min():.2f} .. {S.max():.2f} px   (Z in [{Z_MIN}, {Z_MAX}] m)')
print(f"\n{'阈值':>8s} {'蒙特卡洛':>12s} {'解析式':>12s}")
for thr in [8, 12, 16, 24, 32, 48, 96]:
    emp, ana = float((S < thr).mean()), float(frac_below_analytic(thr))
    assert abs(emp - ana) < 0.005, (thr, emp, ana)
    print(f'  s<{thr:3d}px {emp:11.3%} {ana:11.3%}')

FRAC_SMALL = float((S < 32).mean())
assert FRAC_SMALL > 0.80, FRAC_SMALL
assert float((S < 16).mean()) > 0.55
print(f'\n⚠️  {FRAC_SMALL:.1%} 的实例小于 32 px（COCO 的 small 定义）——')
print('    这不是数据集采集偏好，是 p(s) ∝ 1/s² 的几何必然。')
print('✅ 推导: s = c/Z, Z~U(a,b)  =>  p(s) = p(Z)|dZ/ds| = c / ((b-a) s²)')

In [ ]:
# 标注截断的影响：很多数据集会丢弃 < 8px 的框，这让"数据集看起来"没那么极端
for cut in [0, 4, 8, 12, 16]:
    kept = S[S >= cut]
    print(f'标注下限 {cut:2d} px: 保留 {len(kept)/len(S):6.1%} 的实例, '
          f'其中 <32px 占 {float((kept < 32).mean()):6.1%}, 中位尺寸 {np.median(kept):5.1f} px')
kept8 = S[S >= 8]
assert (kept8 < 32).mean() < FRAC_SMALL, '截断后小目标占比下降'
print('\n⚠️  标注截断是"离线指标好、路测体验差"的一个隐性来源：')
print('    被丢掉的极小目标在路上是**真实存在**的，只是评测集里看不到它们。')
print('    正确做法：把 <8px 的框标成 ignore 区域（不算正也不算负），而不是直接删掉。')

## 2 · FPN 层级分配公式，以及 TSR 分布下的层级负载

In [ ]:
def fpn_level(w, h, k0=4, lo=2, hi=5):
    '''FPN / Mask R-CNN 的 RoI 层级分配： k = floor(k0 + log2(sqrt(wh)/224))，再 clip。'''
    k = k0 + math.floor(math.log2(math.sqrt(w * h) / 224.0 + 1e-12))
    return int(min(max(k, lo), hi))

STRIDE = {2: 4, 3: 8, 4: 16, 5: 32, 6: 64, 7: 128}

print(f"{'sqrt(wh)':>10s} {'log2(s/224)':>12s} {'k(未clip)':>10s} "
      f"{'k(P2-P5)':>10s} {'stride':>7s} {'占格子/轴':>10s}")
for s in [448, 224, 112, 56, 32, 20, 10, 6]:
    raw = 4 + math.floor(math.log2(s / 224.0))
    k = fpn_level(s, s)
    print(f'{s:>10d} {math.log2(s/224.0):>12.2f} {raw:>10d} '
          f'{"P%d" % k:>10s} {STRIDE[k]:>7d} {s/STRIDE[k]:>10.2f}')

assert fpn_level(224, 224) == 4
assert fpn_level(112, 112) == 3
assert fpn_level(56, 56)   == 2
assert fpn_level(500, 500) == 5
assert fpn_level(10, 10)   == 2, '10px 被 clip 到 P2'
assert fpn_level(10, 10, lo=3) == 3, '没有 P2 时 clip 到 P3'
print('\n⚠️  公式在小尺寸上是**饱和**的：所有 <56px 的框都被 clip 到同一层。')
print('    公式没错，它只是在说「我已经没有更高分辨率的层可以给你了」。')

In [ ]:
def level_load(sizes, lo, hi, k0=4):
    '''统计一批目标在金字塔各层上的负载占比。'''
    ks = np.array([fpn_level(s, s, k0=k0, lo=lo, hi=hi) for s in sizes])
    return {int(k): float((ks == k).mean()) for k in range(lo, hi + 1)}

SUB = S[rng.choice(len(S), 40_000, replace=False)]      # 抽样加速
load_p2 = level_load(SUB, lo=2, hi=5)
load_p3 = level_load(SUB, lo=3, hi=5)
load_p37 = level_load(SUB, lo=3, hi=7)

print(f"{'配置':<22s} " + ' '.join(f'{"P%d"%k:>8s}' for k in range(2, 8)))
for name, ld in [('P2-P5 (含 P2)', load_p2), ('P3-P5 (主流)', load_p3),
                 ('P3-P7 (RetinaNet)', load_p37)]:
    cells = ' '.join(f'{ld.get(k, float("nan")):>7.1%}' if k in ld else f'{"—":>8s}'
                     for k in range(2, 8))
    print(f'{name:<22s} {cells}')

assert load_p2[2] > 0.98,  load_p2
assert load_p3[3] > 0.999, load_p3
assert load_p37[3] > 0.999 and load_p37[6] == 0.0 and load_p37[7] == 0.0
print(f'\n⚠️  P2-P5: {load_p2[2]:.1%} 的目标压在 P2；P3-P5/P3-P7: **100%** 压在 P3。')
print('    => 在 TSR 分布下，P3-P7 的 RetinaNet 等价于一个「单尺度 stride 8」检测器。')
print('    你付了 5 层金字塔的钱，只有 1 层在干活，而它的 stride 是 8。')

# 每层的"格子预算"：目标在被分配到的那层上占几个格子
for name, lo in [('有 P2', 2), ('无 P2', 3)]:
    ks = np.array([fpn_level(s, s, lo=lo, hi=5) for s in SUB])
    cells = SUB / np.array([STRIDE[k] for k in ks])
    print(f'{name}: 中位格子数/轴 = {np.median(cells):.2f}, '
          f'低于 1 个格子的比例 = {float((cells < 1).mean()):.1%}')

## 3 · 「有几个中心点落在框里」：正样本稀缺的第一层原因

center-based 分配（FCOS / YOLOX / 大多数 anchor-free 检测器）要求**特征图的格子中心落在 GT 框内**。
对小目标，这个条件本身就可能无法满足——而且概率可以精确算出来。

In [ ]:
def p_zero_center_analytic(size_px, stride):
    '''格子中心位于 {c0 + k*stride}。GT 框边长 L 随机放置时，
       每个轴上落入至少一个中心的概率 = min(1, L/stride)。
       二维需要两个轴同时满足 => P(一个正样本都没有) = 1 - min(1, L/s)^2 。'''
    p_axis = min(1.0, size_px / stride)
    return 1.0 - p_axis ** 2

def p_zero_center_mc(size_px, stride, n=200_000, rng=rng):
    '''蒙特卡洛：框左上角在一个 stride 周期内均匀放置。'''
    a = rng.uniform(0.0, stride, size=(n, 2))
    # 格子中心在 stride/2 + k*stride；数一数 [a, a+L) 内有几个
    lo = np.ceil((a - stride / 2.0) / stride)
    hi = np.ceil((a + size_px - stride / 2.0) / stride)
    cnt = (hi - lo)                       # 每轴的中心点个数
    return float((cnt.min(axis=1) == 0).mean())

print(f"{'目标边长':>9s} {'stride':>7s} {'E[中心点数]':>12s} "
      f"{'P(0个正样本) 解析':>18s} {'蒙特卡洛':>10s}")
for L, s in [(4, 8), (6, 8), (8, 8), (10, 8), (16, 8),
             (4, 4), (6, 4), (10, 4), (10, 16), (10, 32)]:
    ana, mc = p_zero_center_analytic(L, s), p_zero_center_mc(L, s)
    assert abs(ana - mc) < 0.01, (L, s, ana, mc)
    print(f'{L:>9d} {s:>7d} {(L/s)**2:>12.2f} {ana:>17.1%} {mc:>10.1%}')

assert p_zero_center_analytic(6, 8) > 0.4,  '6px 目标在 stride 8 上有 >40% 概率一个正样本都没有'
assert p_zero_center_analytic(6, 4) == 0.0, '换到 stride 4 就必然有正样本'
print('\n⚠️  stride 8 上：4px 目标 75% 没有正样本，6px 目标 44% 没有正样本。')
print('    「没有正样本」= 这个 GT 对损失没有任何贡献 = 模型永远学不到它。')
print('✅ 换到 stride 4（加 P2），>=4px 的目标就必然至少有一个中心点。')

In [ ]:
# 把这个概率套到真实的 TSR 尺寸分布上，算"结构性漏检率"
for lo, name in [(3, '无 P2 (最细 stride 8)'), (2, '有 P2 (最细 stride 4)')]:
    ks = np.array([fpn_level(s, s, lo=lo, hi=5) for s in SUB])
    strides = np.array([STRIDE[k] for k in ks], dtype=float)
    p0 = np.array([p_zero_center_analytic(L, st) for L, st in zip(SUB, strides)])
    print(f'{name:<26s} 结构性零正样本率 = {p0.mean():.2%}')

ks3 = np.array([fpn_level(s, s, lo=3, hi=5) for s in SUB])
p0_no_p2 = np.mean([p_zero_center_analytic(L, STRIDE[k]) for L, k in zip(SUB, ks3)])
ks2 = np.array([fpn_level(s, s, lo=2, hi=5) for s in SUB])
p0_p2 = np.mean([p_zero_center_analytic(L, STRIDE[k]) for L, k in zip(SUB, ks2)])
assert p0_no_p2 > 3 * p0_p2, (p0_no_p2, p0_p2)
print(f'\n✅ 加 P2 把结构性零正样本率从 {p0_no_p2:.2%} 降到 {p0_p2:.2%}（{p0_no_p2/max(p0_p2,1e-9):.1f}×）')
print('   注意这只是**必要条件**——有中心点不等于能匹配上（IoU 阈值还有一关，见模块 03）。')

## 4 · 加 P2 的账：FLOPs 与显存，精确到 85/21

In [ ]:
IN_HW = (864, 1536)      # 车端常用的 16:9 输入

def level_cost(hw, stride, C=128, n_conv=4, roi_frac=1.0, k=3):
    '''该层 head 塔的 MACs。roi_frac: 只在图像上方这一比例的高度上计算（ROI 限制）。'''
    H, W = hw
    pos = (int(H * roi_frac) // stride) * (W // stride)
    return pos * n_conv * k * k * C * C

def act_mem_bytes(hw, stride, C=128, n_tensors=8, bytes_per=2, roi_frac=1.0):
    H, W = hw
    return (int(H * roi_frac) // stride) * (W // stride) * C * n_tensors * bytes_per

print(f"{'层':>4s} {'stride':>7s} {'位置数':>10s} {'相对P3':>8s} "
      f"{'head MACs':>13s} {'激活显存':>11s}")
tot_pos = {}
for lvl, st in [(2, 4), (3, 8), (4, 16), (5, 32)]:
    H, W = IN_HW
    pos = (H // st) * (W // st)
    tot_pos[lvl] = pos
    print(f'P{lvl:<3d} {st:>7d} {pos:>10,d} {4.0**(3-lvl):>8.4f} '
          f'{level_cost(IN_HW, st)/1e9:>11.2f} G {act_mem_bytes(IN_HW, st)/1e6:>9.1f} MB')

base = sum(level_cost(IN_HW, s) for s in (8, 16, 32))
with_p2 = sum(level_cost(IN_HW, s) for s in (4, 8, 16, 32))
ratio = with_p2 / base
print(f'\nP3-P5 合计 {base/1e9:6.2f} G MACs, 位置数 {sum(tot_pos[l] for l in (3,4,5)):,d}')
print(f'P2-P5 合计 {with_p2/1e9:6.2f} G MACs, 位置数 {sum(tot_pos.values()):,d}')
print(f'倍数 = {ratio:.6f}   （解析值 85/21 = {85/21:.6f}）')
assert abs(ratio - 85/21) < 1e-6, ratio
assert abs(tot_pos[2] / tot_pos[3] - 4.0) < 1e-9
print('\n✅ 加一层，检测头计算量 ×4.05 —— 因为 P2 一层就是 P3+P4+P5 之和的 3 倍。')

In [ ]:
# 四种降本手段，以及组合后的真实倍数
# 每个方案写成 [(stride, C, n_conv, roi_frac), ...]
PLANS = {
    '基线 P3-P5':                 [(8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)],
    '朴素加 P2':                  [(4,128,4,1.0), (8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)],
    'P2 通道减半 (C=64)':         [(4, 64,4,1.0), (8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)],
    'P2 头变浅 (1 conv)':         [(4,128,1,1.0), (8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)],
    'P2 只算上半图 (ROI 0.5)':    [(4,128,4,0.5), (8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)],
    'P2 减半通道 + ROI 0.5':      [(4, 64,4,0.5), (8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)],
}
base_cost = sum(level_cost(IN_HW, *L) for L in PLANS['基线 P3-P5'])
print(f"{'方案':<26s} {'总 MACs':>11s} {'倍数':>8s} {'P2 占比':>9s}")
results = {}
for name, plan in PLANS.items():
    tot = sum(level_cost(IN_HW, *L) for L in plan)
    p2c = sum(level_cost(IN_HW, *L) for L in plan if L[0] == 4)
    results[name] = tot / base_cost
    print(f'{name:<26s} {tot/1e9:>9.2f} G {tot/base_cost:>8.3f}x {p2c/tot:>8.1%}')

assert abs(results['朴素加 P2'] - 85/21) < 1e-6
assert abs(results['P2 通道减半 (C=64)'] - results['P2 头变浅 (1 conv)']) < 1e-9, \
    'C 减半(∝C²，÷4) 与 conv 数 4->1(∝n，÷4) 效果相同'
assert results['P2 减半通道 + ROI 0.5'] < 1.45
print(f"\n✅ 朴素加 P2 是 {results['朴素加 P2']:.2f}x，"
      f"减半通道+ROI 后只有 {results['P2 减半通道 + ROI 0.5']:.2f}x。")
print('   心法：**P2 不需要和 P3-P5 同构** —— 它面对的目标只有 4-30 px，')
print('   既不需要 128 通道去区分 200 类的精细语义，也不需要 4 层塔做大范围回归。')

In [ ]:
# 隐性代价：加 P2 会让前景-背景比进一步恶化约 4 倍
N_SIGNS_PER_IMG = 5
POS_PER_GT      = 9          # center-based 分配典型值（由目标尺寸决定，与 stride 关系不大）
n_pos = N_SIGNS_PER_IMG * POS_PER_GT
for name, lvls in [('P3-P5', (3,4,5)), ('P2-P5', (2,3,4,5))]:
    n_total = sum(tot_pos[l] for l in lvls)
    print(f'{name}: 总位置 {n_total:>8,d}, 正样本 {n_pos:>3d}, '
          f'正样本占比 {n_pos/n_total:.4%}, 正负比 1:{n_total/n_pos:,.0f}')
r_p3 = n_pos / sum(tot_pos[l] for l in (3,4,5))
r_p2 = n_pos / sum(tot_pos[l] for l in (2,3,4,5))
assert r_p3 / r_p2 > 3.9
print(f'\n⚠️  加 P2 让正样本占比从 {r_p3:.4%} 掉到 {r_p2:.4%}（恶化 {r_p3/r_p2:.1f}x）。')
print('    你花 4 倍算力买来的分辨率，同时把分类头推到了 1:2400 的正负比。')
print('✅ 结论：**架构改动必须和分配/损失改动一起做** —— 这正是模块 03 的内容。')

# 显存
mem_base = sum(act_mem_bytes(IN_HW, s) for s in (8,16,32))
mem_p2   = sum(act_mem_bytes(IN_HW, s) for s in (4,8,16,32))
print(f'\n激活显存(fp16, 8 个中间张量, batch=1): '
      f'P3-P5 {mem_base/1e6:.1f} MB -> P2-P5 {mem_p2/1e6:.1f} MB ({mem_p2/mem_base:.2f}x)')
assert abs(mem_p2/mem_base - 85/21) < 1e-6

## 5 · 多尺度融合：FPN 的单向性（脉冲实验）、PAN 与 BiFPN 加权融合

In [ ]:
def upsample2x(x):
    '''最近邻 2x 上采样， x: (H, W, C)'''
    return np.repeat(np.repeat(x, 2, axis=0), 2, axis=1)

def downsample2x(x):
    '''2x 平均池化下采样'''
    H, W, C = x.shape
    return x.reshape(H // 2, 2, W // 2, 2, C).mean(axis=(1, 3))

_t = rng.normal(size=(8, 8, 3))
assert np.allclose(downsample2x(upsample2x(_t)), _t), '最近邻上采样后平均池化应严格还原'

def fpn_topdown(feats):
    '''自顶向下：语义从深层广播到浅层。feats: {level: (H,W,C)}，level 越大分辨率越低。'''
    out, prev = {}, None
    for l in sorted(feats, reverse=True):
        cur = feats[l].copy()
        if prev is not None:
            cur = cur + upsample2x(prev)
        out[l] = cur
        prev = cur
    return out

def pan_bottomup(feats):
    '''自底向上：定位信息从浅层送到深层。'''
    out, prev = {}, None
    for l in sorted(feats):
        cur = feats[l].copy()
        if prev is not None:
            cur = cur + downsample2x(prev)
        out[l] = cur
        prev = cur
    return out

SHAPES = {2: (32, 32, 4), 3: (16, 16, 4), 4: (8, 8, 4), 5: (4, 4, 4)}
feats = {l: np.zeros(s) for l, s in SHAPES.items()}
feats[2][10, 10, 0] = 1.0                    # 在 P2 放一个脉冲：细节/定位信息

td = fpn_topdown(feats)
pan = pan_bottomup(td)
print(f"{'层':>4s} {'FPN(top-down) 后能量':>22s} {'再接 PAN(bottom-up) 后':>24s}")
for l in sorted(SHAPES):
    print(f'P{l:<3d} {td[l].sum():>22.4f} {pan[l].sum():>24.4f}')

assert td[2].sum() == 1.0
assert td[5].sum() == 0.0 and td[4].sum() == 0.0 and td[3].sum() == 0.0, \
    'FPN 的 top-down **无法**把 P2 的信息送到深层'
assert pan[5].sum() > 0.0 and pan[3].sum() > 0.0, 'PAN 的 bottom-up 补上了这条通路'
print('\n⚠️  FPN 的信息流是**单向**的：P2 的脉冲在 P3/P4/P5 上响应严格为 0。')
print('    在原始 backbone 里，C2 -> C5 要穿过 100+ 层；PAN 的捷径只有不到 10 层。')
print('✅ 这不是比喻，是可以 assert 的事实。')

In [ ]:
def fast_norm_fusion(inputs, w_raw, eps=1e-4):
    '''BiFPN 的 fast normalized fusion:  O = sum_i ReLU(w_i)/(eps + sum_j ReLU(w_j)) * I_i'''
    w = np.maximum(np.asarray(w_raw, dtype=float), 0.0)
    denom = eps + w.sum()
    out = np.zeros_like(np.asarray(inputs[0], dtype=float))
    for wi, x in zip(w, inputs):
        out = out + wi * np.asarray(x, dtype=float)
    return out / denom

a = rng.normal(size=(4, 4, 2))
b = rng.normal(size=(4, 4, 2))
c = rng.normal(size=(4, 4, 2))

TOL = dict(rtol=1e-3, atol=1e-4)          # eps=1e-4 会带来 ~1e-4 的相对偏差，用 rtol 判定
assert np.allclose(fast_norm_fusion([a, b], [1.0, 1.0]), (a + b) / 2, **TOL), '等权 = 平均'
assert np.allclose(fast_norm_fusion([a, b], [1.0, 0.0]), a, **TOL),          'one-hot = 直通'
assert np.allclose(fast_norm_fusion([a, b], [1.0, -5.0]), a, **TOL),         'ReLU 把负权重归零'
assert np.allclose(fast_norm_fusion([a, b, c], [2.0, 1.0, 1.0]),
                   (2*a + b + c) / 4, **TOL)

print(f"{'原始权重':>22s} {'归一化后':>28s}")
for w_raw in [[1., 1.], [3., 1.], [1., 0.], [5., 1., 1.], [-2., 1., 1.]]:
    w = np.maximum(np.array(w_raw), 0.0)
    norm = w / (1e-4 + w.sum())
    print(f'{str(w_raw):>22s} {str(np.round(norm, 3).tolist()):>28s}')

# 为什么加权对小目标重要：P2 自身证据 vs 上采样来的语义
own   = np.ones((4, 4, 2)) * 1.0        # P2 自身（高分辨率、语义弱、噪声大）
sem   = np.ones((4, 4, 2)) * 4.0        # 从 P3 上采样来的语义（幅值更大）
eq    = fast_norm_fusion([own, sem], [1.0, 1.0])
learn = fast_norm_fusion([own, sem], [3.0, 1.0])
print(f'\n等权相加:   融合值 {eq.mean():.3f}  -> P2 自身证据只占 {1*1/(1+4):.1%} 的能量')
print(f'学到的权重: 融合值 {learn.mean():.3f}  -> P2 自身证据占 {3*1/(3*1+1*4):.1%}')
assert eq.mean() > learn.mean(), '把权重挪给幅值小的 P2 后，融合结果被 P2 拉低'
print('✅ 加权融合让网络自己决定「这一层的证据值多少」，')
print('   而不是默认「自身分辨率证据 = 借来的语义」同等可信。')

## 6 · 可变形采样：token 数的账，与小目标上的覆盖率

In [ ]:
# 多尺度全局 attention 的 token 数
tokens = {l: (IN_HW[0] // STRIDE[l]) * (IN_HW[1] // STRIDE[l]) for l in (2, 3, 4, 5)}
N_KV = sum(tokens.values())
M, L_LV, K = 8, 4, 4                       # 头数 / 层数 / 每层采样点
n_samples = M * L_LV * K

for l in sorted(tokens):
    print(f'P{l}: {IN_HW[0]//STRIDE[l]:>4d} x {IN_HW[1]//STRIDE[l]:>4d} = {tokens[l]:>8,d} token')
print(f'{"总计":<5s}{" ":>17s}{N_KV:>8,d} token')
print(f'\n全局 self-attention: O(N²·C) = {N_KV:,d}² × 256 ≈ {N_KV**2*256:.2e} MAC —— 不可行')
print(f'可变形采样点数     : M×L×K = {M}×{L_LV}×{K} = {n_samples}')
print(f'比值               : {N_KV/n_samples:,.0f}x')
assert N_KV == 110_160, N_KV
assert 800 < N_KV / n_samples < 900
print(f'\n✅ 可变形注意力把 key 数从 {N_KV:,d} 降到 {n_samples}，约 {N_KV/n_samples:.0f} 倍。')

In [ ]:
def _phi(z):
    '''标准正态 CDF'''
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def sample_coverage(size_px, stride, sigma_cells=1.0):
    '''采样点偏移 ~ N(0, (sigma_cells*stride)²)（Deformable DETR 的偏移初始化尺度 ∝ stride）。
       返回采样点落进 size_px × size_px 目标框内的概率。'''
    sigma = sigma_cells * stride
    p_axis = 2.0 * _phi(size_px / (2.0 * sigma)) - 1.0
    return p_axis ** 2

print(f"{'目标尺寸':>9s} {'层':>4s} {'stride':>7s} {'占格子/轴':>10s} {'采样点命中率':>13s}")
for size in [10, 20]:
    for lvl in (2, 3, 4, 5):
        st = STRIDE[lvl]
        print(f'{size:>9d} {"P%d"%lvl:>4s} {st:>7d} {size/st:>10.2f} '
              f'{sample_coverage(size, st):>12.1%}')
    print()

cov = [sample_coverage(10, STRIDE[l]) for l in (2, 3, 4, 5)]
assert cov[0] > cov[1] > cov[2] > cov[3], '越高分辨率的层，采样点越容易落进小目标'
assert cov[0] > 0.55 and cov[0] / cov[3] > 20
print(f'✅ 10 px 目标: P2 命中率 {cov[0]:.1%}，P5 只有 {cov[3]:.2%}（差 {cov[0]/cov[3]:.0f} 倍）')

# 与"全局均匀注意力"对比：目标能拿到多少注意力质量
cells_on_p2 = (10 / STRIDE[2]) ** 2
uniform_mass = cells_on_p2 / N_KV
print(f'\n全局均匀 attention: 10px 目标在 P2 上占 {cells_on_p2:.2f} 格 / {N_KV:,d} token'
      f' = {uniform_mass:.2e} 的注意力质量')
print(f'可变形采样        : {cov[0]:.2%} 的采样点直接落在目标上')
assert cov[0] / uniform_mass > 1e3
print(f'✅ 相差 {cov[0]/uniform_mass:.1e} 倍 —— 这正是 DETR 收敛慢、Deformable DETR 快的量化根因。')
print('⚠️  实现细节：偏移必须初始化成围绕参考点的规则网格（预测层权重置 0、偏置置为网格），')
print('    随机初始化会让采样点散到全图，小目标上根本收不回来 —— 论文一句话，复现三天。')

## ✏️ 练习 1：层级负载统计器

实现 `fpn_level_load(sizes, k0=4, lo=2, hi=5)`，返回 `{level: 占比}`（所有 level 都要出现，
没有目标的层占比为 0.0）。用上面的 `fpn_level` 即可。

In [ ]:
def fpn_level_load(sizes, k0=4, lo=2, hi=5):
    # TODO: 对每个 size 调用 fpn_level(size, size, k0, lo, hi)，统计各层占比
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
r = fpn_level_load([224, 224, 112, 56], lo=2, hi=5)
assert set(r) == {2, 3, 4, 5}, r
assert abs(r[4] - 0.5) < 1e-9 and abs(r[3] - 0.25) < 1e-9 and abs(r[2] - 0.25) < 1e-9
assert r[5] == 0.0
r2 = fpn_level_load(SUB, lo=2, hi=5)
assert r2[2] > 0.98 and abs(sum(r2.values()) - 1.0) < 1e-9
r3 = fpn_level_load(SUB, lo=3, hi=7)
assert r3[3] > 0.999 and r3[6] == 0.0 and r3[7] == 0.0
print('TSR 分布下的层级负载:')
for name, ld in [('P2-P5', r2), ('P3-P7', r3)]:
    print(f'  {name}: ' + '  '.join(f'P{k}={v:.1%}' for k, v in sorted(ld.items())))
print('✅ 练习 1 通过：**先算负载，再谈方案** —— 这是面试里的分水岭')

## ✏️ 练习 2：金字塔成本比计算器

实现 `plan_cost_ratio(hw, base_plan, new_plan)`，plan 是 `[(stride, C, n_conv, roi_frac), ...]`，
返回 `new_plan` 相对 `base_plan` 的总 MACs 倍数。直接复用上面的 `level_cost`。

In [ ]:
def plan_cost_ratio(hw, base_plan, new_plan):
    # TODO: 分别求和再相除
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
BASE = [(8,128,4,1.0), (16,128,4,1.0), (32,128,4,1.0)]
assert abs(plan_cost_ratio(IN_HW, BASE, BASE) - 1.0) < 1e-12
assert abs(plan_cost_ratio(IN_HW, BASE, [(4,128,4,1.0)] + BASE) - 85/21) < 1e-6
assert abs(plan_cost_ratio(IN_HW, BASE, [(4,64,4,1.0)] + BASE)
           - plan_cost_ratio(IN_HW, BASE, [(4,128,1,1.0)] + BASE)) < 1e-9
r_cheap = plan_cost_ratio(IN_HW, BASE, [(4,64,4,0.5)] + BASE)
assert 1.30 < r_cheap < 1.45, r_cheap
print(f"{'方案':<28s} {'倍数':>8s}")
for name, plan in PLANS.items():
    print(f'{name:<28s} {plan_cost_ratio(IN_HW, BASE, plan):>7.3f}x')
print('✅ 练习 2 通过：**P2 不必与 P3-P5 同构** —— 4.05x 可以压到 1.38x')

## ✏️ 练习 3：BiFPN 的加权融合（含尺寸对齐）

实现 `bifpn_node(inputs, w_raw, target_hw, eps=1e-4)`：
先把每个输入用 `upsample2x` / `downsample2x` 反复调整到 `target_hw`（只会差 2 的整数次幂），
再做 fast normalized fusion。假设所有输入的通道数一致。

In [ ]:
def bifpn_node(inputs, w_raw, target_hw, eps=1e-4):
    # TODO: ① 逐个 resize 到 target_hw（大了就 downsample2x，小了就 upsample2x）
    #       ② 调用 fast_norm_fusion
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
x_hi  = rng.normal(size=(16, 16, 2))
x_mid = rng.normal(size=( 8,  8, 2))
x_lo  = rng.normal(size=( 4,  4, 2))

o = bifpn_node([x_hi, x_mid, x_lo], [1., 1., 1.], (8, 8))
assert o.shape == (8, 8, 2), o.shape
ref = (downsample2x(x_hi) + x_mid + upsample2x(x_lo)) / 3.0
assert np.allclose(o, ref, rtol=1e-3, atol=1e-4)

o1 = bifpn_node([x_hi, x_lo], [1., 0.], (16, 16))
assert o1.shape == (16, 16, 2) and np.allclose(o1, x_hi, rtol=1e-3, atol=1e-4)
o2 = bifpn_node([x_mid], [2.0], (8, 8))
assert np.allclose(o2, x_mid, rtol=1e-3, atol=1e-4), \
    '单输入节点做的只是一次恒等变换 —— BiFPN 把它删掉了'
print('融合输出形状:', o.shape, '| 等权融合与手算参考一致 ✅')
print('✅ 练习 3 通过：加权融合让 P2 的自身证据不被上采样来的语义稀释')

## ✏️ 练习 4：反推 stride —— 「要看清这个目标，最粗能用多大的 stride」

实现 `max_stride_for(size_px, min_cells_per_axis, candidates=(1,2,4,8,16,32,64,128))`：
返回满足 `size_px / stride >= min_cells_per_axis` 的**最大**候选 stride；无解返回 `None`。
再实现 `required_input_scale(native_px, min_cells, stride)`：
在给定 stride 下，要让原生尺寸为 `native_px` 的目标占到 `min_cells` 个格子，
输入图像最少要保留原始分辨率的多少比例。

In [ ]:
def max_stride_for(size_px, min_cells_per_axis, candidates=(1,2,4,8,16,32,64,128)):
    # TODO
    raise NotImplementedError

def required_input_scale(native_px, min_cells, stride):
    # TODO: 返回 min_cells * stride / native_px
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert max_stride_for(10, 2) == 4      # 10/4 = 2.5 >= 2, 而 10/8 = 1.25 < 2
assert max_stride_for(10, 3) == 2
assert max_stride_for(6, 2)  == 2
assert max_stride_for(100, 2) == 32
assert max_stride_for(1, 2) is None
assert abs(required_input_scale(20, 2, 8) - 0.8) < 1e-12
assert abs(required_input_scale(20, 2, 4) - 0.4) < 1e-12

print(f"{'距离':>6s} {'原生px':>8s} {'最粗stride(>=2格)':>18s} "
      f"{'stride8 需保留比例':>20s} {'stride4 需保留比例':>20s}")
for z in [30, 50, 80, 100, 130]:
    s_native = float(sign_px(z))
    ms = max_stride_for(s_native, 2)
    r8 = required_input_scale(s_native, 2, 8)
    r4 = required_input_scale(s_native, 2, 4)
    print(f'{z:>5d}m {s_native:>8.1f} {str(ms):>18s} '
          f'{min(r8,9.99):>19.2f}x {min(r4,9.99):>19.2f}x')
print('\n⚠️  100 m 外的标志（10 px）在 stride 8 上需要**超过原生分辨率**才能占到 2 个格子。')
print('✅ 练习 4 通过：这个求解器把「要检多远」直接翻译成「输入分辨率 + stride」的配置约束')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def fpn_level_load(sizes, k0=4, lo=2, hi=5):
    ks = [fpn_level(s, s, k0=k0, lo=lo, hi=hi) for s in sizes]
    n = max(len(ks), 1)
    return {k: sum(1 for v in ks if v == k) / n for k in range(lo, hi + 1)}

In [ ]:
# 练习 2 参考答案
def plan_cost_ratio(hw, base_plan, new_plan):
    base = sum(level_cost(hw, *L) for L in base_plan)
    new  = sum(level_cost(hw, *L) for L in new_plan)
    return new / base

In [ ]:
# 练习 3 参考答案
def bifpn_node(inputs, w_raw, target_hw, eps=1e-4):
    aligned = []
    for x in inputs:
        y = np.asarray(x, dtype=float)
        while y.shape[0] > target_hw[0]:
            y = downsample2x(y)
        while y.shape[0] < target_hw[0]:
            y = upsample2x(y)
        aligned.append(y)
    return fast_norm_fusion(aligned, w_raw, eps=eps)

In [ ]:
# 练习 4 参考答案
def max_stride_for(size_px, min_cells_per_axis, candidates=(1,2,4,8,16,32,64,128)):
    ok = [s for s in candidates if size_px / s >= min_cells_per_axis]
    return max(ok) if ok else None

def required_input_scale(native_px, min_cells, stride):
    return min_cells * stride / float(native_px)

---
## 🧪 真实工程胶囊：把本模块的结论变成配置

In [ ]:
RECIPE = r'''
# ============================================================================
# TSR 小目标 · 架构层面的配置清单（按收益/成本排序，从上往下做）
# ============================================================================

# ---- 步骤 0：先量化，别急着改模型 -------------------------------------------
#   ① 统计标注框的 sqrt(w*h) 分布（画直方图 + 5/25/50/75/95 分位）
#   ② 算每个框在"它会被分配到的那一层"上占几个格子： size / stride
#   ③ 算层级负载： 各层的目标占比。>90% 集中在一层 = 金字塔在空转
#   ④ 算 letterbox 缩放比例： imgsz / 原图宽。这一步常常就能找到问题。

# ---- 步骤 1：把输入分辨率改回接近原生（最高 ROI，零工程风险）-----------------
# Ultralytics YOLO
model.train(data='tsr.yaml',
            imgsz=1280,            # ← 从默认 640 提到 1280；1920 原生更好但要看延迟
            rect=True,             # 保持 16:9，别 pad 成正方形（pad 是纯浪费的算力）
            scale=0.5, mosaic=1.0) # 多尺度训练下界不要太低，否则更伤小目标

# MMDetection：train_pipeline / test_pipeline 的 scale 必须一致
train_pipeline = [
    dict(type='Resize', scale=(1536, 864), keep_ratio=True),   # ← 不是 (1333, 800)
    ...
]

# ---- 步骤 2：加 P2，但让它便宜 ------------------------------------------------
# MMDetection: FPN 从 P2 开始（start_level=0 表示用 C2）
neck = dict(
    type='FPN',
    in_channels=[256, 512, 1024, 2048],   # C2..C5
    out_channels=128,                      # ← 车端用 128 而不是 256
    start_level=0,                         # ← 0 = 从 C2 开始，产出 P2
    num_outs=4,                            # P2..P5（不要 P6/P7，TSR 上它们负载为 0）
)
bbox_head = dict(
    strides=[4, 8, 16, 32],                # ← 必须同步改！忘了改是最常见的坑
    stacked_convs=4,
    # anchor-based 时还要改 anchor 尺度：让最小 anchor 落在尺寸分布的 5% 分位
    anchor_generator=dict(type='AnchorGenerator',
                          octave_base_scale=2,     # ← 默认 4 => P2 最小 anchor 16px，太大
                          scales_per_octave=3,
                          ratios=[0.8, 1.0, 1.25], # 交通标志接近方形，别用 [0.5,1,2]
                          strides=[4, 8, 16, 32]),
)
# FCOS/ATSS 风格还要改回归范围：
#   regress_ranges = ((-1, 32), (32, 96), (96, 256), (256, 1e8))   # ← 为小目标下移

# ---- 步骤 3：neck 换加权融合（成本小，小模型上收益明显）----------------------
neck = dict(type='BiFPN' , num_stages=3, out_channels=128, start_level=0)
# 或 YOLO 系直接用 PAN（v5/v8 默认已含），确认它确实接了 P2 分支

# ---- 步骤 4：上下文与位置先验（几乎零成本）-----------------------------------
#   · 主干里换 5x5 depthwise 大核（RTMDet 的做法），扩大有效感受野
#   · 把归一化坐标 (x/W, y/H) 作为额外 2 个通道拼进 neck 输入
#   · 训练时对位置先验加扰动增强（随机上下平移），防止学成"位置捷径"

# ---- 步骤 5：评测必须按尺寸分桶，否则改进不可见 ------------------------------
SIZE_BUCKETS = [(0, 8), (8, 16), (16, 32), (32, 96), (96, 1e9)]
# 只看总 mAP 的话，一个"AP_s +4 / AP_l -1"的正确改动可能显示为 +0.3（看起来像噪声）

# ---- 反模式清单（看到就要警惕）-----------------------------------------------
#   ✗ imgsz=640 训 1920 的图，然后抱怨小目标检不到
#   ✗ 加了 P2 但没改 strides / anchor 尺度 / regress_ranges
#   ✗ P3-P7 的配置用在 TSR 上（P6/P7 永远拿不到目标）
#   ✗ 用总 mAP 做剪枝/量化的敏感度评估（会系统性牺牲小目标）
#   ✗ 上超分：它不创造信息，它发明信息 —— "60" 可能被补成 "80"
'''
print(RECIPE)
for key in ['imgsz=1280', 'start_level=0', 'strides=[4, 8, 16, 32]',
            'octave_base_scale', 'regress_ranges', 'SIZE_BUCKETS', '反模式']:
    assert key in RECIPE, key
print('✅ 配方覆盖：先量化 -> 提分辨率 -> 便宜地加 P2 -> 加权融合 -> 上下文 -> 分桶评测')

### 小结

- **层级分配公式 `k = ⌊k0 + log2(√(wh)/224)⌋` 在小尺寸上是饱和的**：所有 <56 px 的框被 clip 到同一层。
  在 TSR 的尺寸分布下，**P3–P7 的金字塔有 100% 的目标压在 P3**——你付了 5 层的钱，只有 1 层在干活。
- **TSR 的小目标占比是几何必然，不是数据集偏差**：s = fS/Z 加上距离均匀采样 ⇒ p(s) ∝ 1/s²，
  于是 84% 的实例 <32 px。**推导它比统计它更有说服力。**
- **加 P2 的成本恰好是 85/21 ≈ 4.05×**（neck+head），显存同比例。但 **P2 不必与其他层同构**：
  通道减半（∝C²，÷4）或头变浅（÷4）各能把倍数压到 1.76×，配合 ROI 限制可到 **1.38×**。
- **加 P2 的隐性代价是正负样本比恶化 4 倍**（0.165% → 0.041%）。
  **架构改动必须和分配/损失改动一起做**——这是模块 03 的内容。
- **提分辨率 ≠ 加 P2 ≠ 换主干**：「目标的像素数」决定信息量上界，「占的格子数」决定网络能否表示它。
  **提分辨率两项都动，加 P2 只动第二项，换主干两项都不动**——这解释了「换了更强 backbone，AP_s 纹丝不动」。
- **FPN 的信息流是单向的**（P2 的脉冲在 P5 上响应严格为 0），PAN 补上自底向上的捷径，
  BiFPN 再加上**可学的加权融合**，让 P2 的自身证据不被上采样来的语义稀释。
- **可变形注意力把 key 数从 110,160 降到 128（861×）**，并让采样点命中小目标的比例从
  5.7×10⁻⁵ 提到 62%——但**偏移初始化必须是规则网格**，随机初始化直接不收敛。
- **上下文范围取目标尺寸的 3–6 倍**：太小拿不到杆件，太大会学成「路边 = 有标志」的捷径 → 广告牌误检。
- **超分是最后手段且有安全风险**：它不创造信息，它发明信息。8 px 的「60」可能被补成清晰自信的「80」。

下一站：**模块 03 · 分配与损失层面的解法**——当分辨率已经用足，
剩下的问题是「IoU 这把尺子本身对小框就是坏的」，而 **NWD** 换掉了这把尺子。